# NVDEC Build Test — ffmpeg + PyAV Only (Colab)

Standalone test of just the risky/slow part: compiling ffmpeg from source
with CUDA/NVDEC support, then rebuilding PyAV against it. No project code,
no video, no TensorRT — purely answering one question: **can this build
succeed on Colab at all?**

If step 4 shows `cuda` with `is_supported=True`, the build worked and
you can move on to the full `VisionEdge_NVDEC_Build_Test.ipynb` notebook
(project + video + actual pipeline test) with confidence. If it doesn't,
you've found that out in ~30 minutes instead of after also uploading
your whole project.

Before running: **Runtime -> Change runtime type -> T4 GPU -> Save.**

## 0. Confirm GPU + CUDA version

In [1]:
!nvidia-smi
!nvcc --version

Thu Aug  6 09:37:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install build tools + NVIDIA's codec headers

`nv-codec-headers` tells ffmpeg's build system how to talk to NVDEC —
required before configuring the ffmpeg build in step 2.

In [2]:
!apt-get update -qq
!apt-get install -y -qq autoconf automake build-essential cmake git-core \
    libtool pkg-config texinfo wget yasm zlib1g-dev nasm

!git clone https://github.com/FFmpeg/nv-codec-headers.git
%cd nv-codec-headers
!make install
%cd ..

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
(Reading database ... 122403 files and directories currently installed.)
Removing r-base-dev (4.6.0-4.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libglib2.0-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libfontconfig-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which 

## 2. Compile ffmpeg from source with CUDA/NVDEC enabled

**This is the slow step.** Grab a coffee — this can take 20-40+ minutes
on a Colab T4 instance. If `./configure` itself fails, read the error
carefully before re-running — it usually names the exact missing tool
or header.

In [12]:
!git clone --depth 1 https://github.com/FFmpeg/FFmpeg.git ffmpeg_src
%cd ffmpeg_src
!make distclean
!./configure \
    --prefix=/content/ffmpeg_build \
    --enable-shared \
    --disable-static \
    --enable-cuda-nvcc \
    --enable-cuvid \
    --enable-nvdec \
    --enable-nvenc \
    --enable-nonfree \
    --enable-libnpp \
    --extra-cflags=-I/usr/local/cuda/include \
    --extra-ldflags=-L/usr/local/cuda/lib64
!make -j$(nproc)
!make install
%cd ..

fatal: destination path 'ffmpeg_src' already exists and is not an empty directory.
/ffmpeg_src
install prefix            /content/ffmpeg_build
source path               .
C compiler                gcc
C library                 glibc
ARCH                      x86 (generic)
big-endian                no
runtime cpu detection     yes
standalone assembly       yes
x86 assembler             nasm
MMX enabled               yes
MMXEXT enabled            yes
SSE enabled               yes
SSSE3 enabled             yes
AESNI enabled             yes
CLMUL enabled             yes
AVX enabled               yes
AVX2 enabled              yes
AVX-512 enabled           yes
AVX-512ICL enabled        yes
XOP enabled               yes
FMA3 enabled              yes
FMA4 enabled              yes
i686 features enabled     yes
CMOV is fast              yes
EBX available             yes
6 registers available     yes
7 registers available     yes
debug symbols             yes
strip symbols             yes
optimiz

In [13]:
!git ls-remote https://github.com/FFmpeg/FFmpeg.git HEAD

95c43d7df7b72e3a4e8dce8c6718cffedb32211d	HEAD


In [15]:
!which ffmpeg
!ffmpeg -version
!pkg-config --modversion libavcodec
!pkg-config --modversion libavformat
!pkg-config --modversion libavutil
!ls /content/ffmpeg_build/lib | grep avcodec

libavcodec.a
libavcodec.so
libavcodec.so.63
libavcodec.so.63.7.100


In [17]:
!find /content/ffmpeg_build -name "*.pc"

/content/ffmpeg_build/lib/pkgconfig/libswresample.pc
/content/ffmpeg_build/lib/pkgconfig/libavutil.pc
/content/ffmpeg_build/lib/pkgconfig/libavdevice.pc
/content/ffmpeg_build/lib/pkgconfig/libavfilter.pc
/content/ffmpeg_build/lib/pkgconfig/libswscale.pc
/content/ffmpeg_build/lib/pkgconfig/libavcodec.pc
/content/ffmpeg_build/lib/pkgconfig/libavformat.pc


In [18]:
!echo $PKG_CONFIG_PATH
!ls /content/ffmpeg_build/lib/pkgconfig

libavcodec.pc	libavfilter.pc	libavutil.pc	  libswscale.pc
libavdevice.pc	libavformat.pc	libswresample.pc


## 3. Build PyAV from source against your custom ffmpeg

In [21]:
%%bash

export PATH=/content/ffmpeg_build/bin:$PATH
export LD_LIBRARY_PATH=/content/ffmpeg_build/lib:$LD_LIBRARY_PATH
export PKG_CONFIG_PATH=/content/ffmpeg_build/lib/pkgconfig

pip uninstall -y av

pip install av --no-binary av --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 85.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for av: filename=av-18.0.0-cp311-abi3-linux_x86_64.whl size=8487373 sha256=7d58ab37c6eeabfe23627edf738f6df2e5adf71a31c28e915c404e7f2ab8f4a8
  Stored in directory: /tmp/pip-ephem-wheel-cache-i0sutn0l/wheels/df/68/a2/b46ccc0c0eb8c938bbc83efc3f8c5b78d3d0cd005cbf9c23bb
Successfully built av


## 4. Verify it actually worked

The real test — don't skip this. Look for a `cuda` entry with
`is_supported=True` in the printed output.

In [22]:
!python -m av --hwconfigs

Hardware configs:
    av1
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x7e573291b910>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x7e573291b8f0>
    av1_cuvid
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x7e57325f91e0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x7e57325f91c0>
    h263
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x7e5732924fd0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x7e5732924fb0>
    h263p
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x7e5732924fd0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x7e5732924fb0>
    h264
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x7e57329250d0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x7e57329250b0>
    h264_cuvid
        <av.HWConfig device_type

## Result

- **If you saw `cuda` with `is_supported=True`**: the build works on
  Colab. Save it to Drive so you don't have to redo this (cell below),
  then move on to the full `VisionEdge_NVDEC_Build_Test.ipynb` notebook
  to test it against the actual project.
- **If you didn't**: scroll back up through step 2 and 3's output for the
  actual error — paste it back for help debugging, rather than re-running
  blind. Falling back to `test_zero_copy_gpu.py` (a separate, already-
  working notebook) remains solid, credible proof on its own regardless.

## (Optional) Save the successful build to Google Drive

Avoids repeating steps 1-3 next session.

In [23]:
from google.colab import drive
drive.mount('/content/drive')
!tar -czf /content/drive/MyDrive/ffmpeg_cuda_build.tar.gz /content/ffmpeg_build
print("Saved. Next session, restore with:")
print("  !tar -xzf /content/drive/MyDrive/ffmpeg_cuda_build.tar.gz -C /")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tar: Removing leading `/' from member names
Saved. Next session, restore with:
  !tar -xzf /content/drive/MyDrive/ffmpeg_cuda_build.tar.gz -C /
